# PatchPilot — 01 · Fault-localization ranker (A1)

Fine-tunes a code encoder (**UniXcoder-base**, 125M parameters) as a **cross-encoder**: given a bug
report and one candidate (a file skeleton or a function), it predicts whether the candidate must be
edited to fix the bug. At inference PatchPilot uses it to re-rank the BM25 top-50 files and then the
functions inside the best files.

**Runtime:** Kaggle notebook · Accelerator **GPU T4 x1** (or P100) · **Internet on** ·
secret **`HF_TOKEN`** (a Hugging Face *write* token) added under *Add-ons → Secrets*.

**Input:** the leakage-checked dataset built by `scripts/build_localization_data.py`, downloaded from
the Hugging Face Hub (`HF_DATASET` below).

**Outputs (pushed to `HF_MODEL`):** model + tokenizer, `training_log.json`, `val_metrics.json`,
`loss_curve.png`, model card. The *reported* metrics are recomputed locally by
`scripts/eval_localization.py` from the pushed model.

In [ ]:
# ---- Configuration: set HF_USER; everything else can stay ----
HF_USER = "YOUR_HF_USERNAME"
HF_DATASET = f"{HF_USER}/patchpilot-localization"
HF_MODEL = f"{HF_USER}/patchpilot-fl-cross-encoder"
PATCHPILOT_REF = "data/localization"  # git branch/tag of PatchPilot to install

BASE_MODEL = "microsoft/unixcoder-base"
MAX_LEN = 384  # tokens per (query, candidate) pair
QUERY_TOKENS = 192  # must match patchpilot.localization.cross_encoder
NEG_FILES = 4  # file negatives per instance (BM25 hard negatives)
NEG_UNITS = 8  # function negatives per instance (same-file first)
EPOCHS = 1
LR = 2e-5
BATCH = 16
GRAD_ACCUM = 2
SEED = 42

In [ ]:
%%capture
!pip install -q "transformers>=4.44" "accelerate>=0.33" "huggingface_hub>=0.24" matplotlib
!pip install -q "git+https://github.com/Tejaswiniprabhakaran/patchpilot@{PATCHPILOT_REF}"

In [ ]:
import json
import os
import random

import numpy as np
import torch
from huggingface_hub import HfApi, hf_hub_download, login

try:  # Kaggle
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:  # Colab
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
login(token=os.environ["HF_TOKEN"])

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
assert torch.cuda.is_available(), "Turn on the GPU accelerator in the notebook settings"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

from patchpilot.localization.data import read_examples

paths = {
    s: Path(hf_hub_download(HF_DATASET, f"{s}.jsonl.gz", repo_type="dataset"))
    for s in ("train", "val", "test")
}
train = list(read_examples(paths["train"]))
val = list(read_examples(paths["val"]))
print(len(train), "train instances,", len(val), "val instances")

In [ ]:
def make_pairs(examples, rng):
    pairs = []
    for ex in examples:
        pos_f = [c for c in ex.files if c.label]
        neg_f = [c for c in ex.files if not c.label]
        pos_u = [c for c in ex.units if c.label]
        gold_paths = set(ex.gold_files)
        same_file = [c for c in ex.units if not c.label and c.cid.split("::")[0] in gold_paths]
        other = [c for c in ex.units if not c.label and c.cid.split("::")[0] not in gold_paths]
        neg_u = rng.sample(same_file, min(NEG_UNITS, len(same_file)))
        neg_u += rng.sample(other, min(NEG_UNITS - len(neg_u), len(other)))
        for c in pos_f + rng.sample(neg_f, min(NEG_FILES, len(neg_f))) + pos_u + neg_u:
            pairs.append((ex.query, c.text, c.label))
    rng.shuffle(pairs)
    return pairs


train_pairs = make_pairs(train, random.Random(SEED))
print(len(train_pairs), "training pairs,", sum(p[2] for p in train_pairs), "positive")

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

tok = AutoTokenizer.from_pretrained(BASE_MODEL)


def short(q):  # identical to CrossEncoderRanker: the query is cut to QUERY_TOKENS tokens
    return tok.decode(tok(q, add_special_tokens=False)["input_ids"][:QUERY_TOKENS])


class PairDataset(torch.utils.data.Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        q, t, y = self.pairs[i]
        enc = tok(short(q), t, truncation="only_second", max_length=MAX_LEN)
        enc["labels"] = y
        return enc


model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2)
args = TrainingArguments(
    output_dir="ckpt",
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_ratio=0.06,
    weight_decay=0.01,
    fp16=True,
    logging_steps=50,
    save_strategy="no",
    report_to="none",
    seed=SEED,
    dataloader_num_workers=2,
)
trainer = Trainer(model=model, args=args, train_dataset=PairDataset(train_pairs), tokenizer=tok)
result = trainer.train()
print(result)
json.dump(trainer.state.log_history, open("training_log.json", "w"), indent=1)
print("peak GPU memory (GB):", round(torch.cuda.max_memory_allocated() / 1e9, 2))

In [ ]:
import matplotlib.pyplot as plt

steps = [h["step"] for h in trainer.state.log_history if "loss" in h]
loss = [h["loss"] for h in trainer.state.log_history if "loss" in h]
plt.figure(figsize=(6, 3.5))
plt.plot(steps, loss)
plt.xlabel("optimizer step")
plt.ylabel("training loss")
plt.title("Cross-encoder fine-tuning")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("loss_curve.png", dpi=150)
plt.show()

In [ ]:
# Validation re-ranking check (monitoring only; reported numbers come from eval_localization.py)
from patchpilot.localization.metrics import first_hit_rank, summarise

model.eval()


@torch.no_grad()
def score(q, texts):
    out = []
    for i in range(0, len(texts), 64):
        chunk = texts[i : i + 64]
        enc = tok(
            [short(q)] * len(chunk),
            chunk,
            truncation="only_second",
            max_length=MAX_LEN,
            padding=True,
            return_tensors="pt",
        ).to(model.device)
        out += torch.softmax(model(**enc).logits.float(), -1)[:, 1].tolist()
    return out


def ranked(ex_items, scores):
    return [c.cid for _, c in sorted(zip(scores, ex_items), key=lambda p: -p[0])]


file_ranks, unit_ranks = [], []
for ex in val:
    file_ranks.append(
        first_hit_rank(ranked(ex.files, score(ex.query, [c.text for c in ex.files])), ex.gold_files)
    )
    if ex.gold_units and ex.units:
        unit_ranks.append(
            first_hit_rank(
                ranked(ex.units, score(ex.query, [c.text for c in ex.units])), ex.gold_units
            )
        )
val_metrics = {"file_level": summarise(file_ranks), "function_level": summarise(unit_ranks)}
print(json.dumps(val_metrics, indent=2))
json.dump(val_metrics, open("val_metrics.json", "w"), indent=2)

In [ ]:
card = f"""---
license: apache-2.0
base_model: {BASE_MODEL}
tags: [fault-localization, cross-encoder, code, patchpilot]
datasets: [{HF_DATASET}]
---
# PatchPilot fault-localization cross-encoder

Fine-tuned from `{BASE_MODEL}` to score (bug report, code candidate) pairs; a candidate is a file
skeleton (path + class/function signatures) or a function. Used by
[PatchPilot](https://github.com/Tejaswiniprabhakaran/patchpilot) to re-rank BM25 candidates.

* Training data: `{HF_DATASET}` (SWE-bench *train* split; its repositories are disjoint from
  SWE-bench Lite, see `results/leakage_check.json` in the PatchPilot repository).
* {len(train_pairs)} training pairs, {EPOCHS} epoch(s), lr {LR}, batch {BATCH}x{GRAD_ACCUM},
  max length {MAX_LEN}, fp16, seed {SEED}.
* Validation re-ranking (monitoring only): `val_metrics.json`. Reported results are in the
  PatchPilot repository under `results/localization/`.
* Limitations: Python only; trained on issue text, not failing-test output; files are seen only
  through their signatures.
"""
open("README.md", "w").write(card)
model.save_pretrained("model")
tok.save_pretrained("model")
for f in ["README.md", "training_log.json", "val_metrics.json", "loss_curve.png"]:
    os.replace(f, f"model/{f}")
api = HfApi()
api.create_repo(HF_MODEL, exist_ok=True)
api.upload_folder(
    folder_path="model",
    repo_id=HF_MODEL,
    commit_message="PatchPilot fault-localization cross-encoder",
)
print("pushed to https://huggingface.co/" + HF_MODEL)